In [1]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


In [29]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
import os
from langchain.agents.middleware import HumanInTheLoopMiddleware

load_dotenv(override=True)


True

In [30]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_completion_tokens=200
)


In [31]:
from langchain_core.tools import tool

@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day."""
    return f"Meeting booked with {person} on {day}."

approval_agent = create_agent(
    model=llm,
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting": True})],
    checkpointer=InMemorySaver()
)

In [33]:
config = {"configurable": {"thread_id": "approval-demo"}}

In [34]:
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]},config
)



In [ ]:
while True:
    user_input = input("Decision [approve / edit / reject / exit]: ").strip().lower()
    if user_input == "exit":
        break
    if user_input not in {"approve", "edit", "reject"}:
        print("Invalid choice. Use approve, edit, reject, or exit.")
        continue

    decision = {"type": user_input}

    if user_input == "edit":
        edited_person = input("New person name: ").strip() or "Sam"
        edited_day = input("New meeting day: ").strip() or "Friday"
        decision["edited_action"] = {
            "name": "book_meeting",
            "args": {"person": edited_person, "day": edited_day},
        }
    elif user_input == "reject":
        decision["message"] = "User rejected the meeting request."

    resumed = await approval_agent.ainvoke(
        Command(resume={"decisions": [decision]}),
        config,
    )
    print(resumed["messages"][-1].content)